# Overfitting Prevention

This notebook applies regularization techniques to the selected neural network architecture to improve generalization and reduce overfitting.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
data_path = Path("../data/Datasets.xlsx")

df = pd.read_excel(
    data_path,
    sheet_name="Recruitment"
)

X = df.drop(columns=["Candidate_ID", "Status"])
y = df["Status"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [5]:
numerical_features = [
    "Experience",
    "Tech_Score",
    "Interview"
]

categorical_features = [
    "Position"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

input_size = X_train_processed.shape[1]

In [7]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [8]:
early_model = Sequential([
    tf.keras.Input(shape=(input_size,)),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(3, activation="softmax")
])

early_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [9]:
early_history = early_model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=200,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=0
)

In [10]:
print("Epochs actually trained:", len(early_history.history["loss"]))

Epochs actually trained: 85


In [11]:
early_train_loss, early_train_acc = early_model.evaluate(
    X_train_processed,
    y_train,
    verbose=0
)

early_val_loss, early_val_acc = early_model.evaluate(
    X_val_processed,
    y_val,
    verbose=0
)

print(f"Training Accuracy: {early_train_acc * 100:.2f}%")
print(f"Validation Accuracy: {early_val_acc * 100:.2f}%")
print(f"Training Loss: {early_train_loss:.4f}")
print(f"Validation Loss: {early_val_loss:.4f}")

Training Accuracy: 78.57%
Validation Accuracy: 60.00%
Training Loss: 0.4398
Validation Loss: 0.5294


In [12]:
dropout_model = Sequential([
    tf.keras.Input(shape=(input_size,)),

    Dense(32, activation="relu"),
    Dropout(0.30),

    Dense(16, activation="relu"),
    Dropout(0.20),

    Dense(3, activation="softmax")
])

In [13]:
dropout_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [14]:
dropout_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [15]:
dropout_history = dropout_model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=200,
    batch_size=16,
    callbacks=[dropout_early_stopping],
    verbose=0
)

In [16]:
print(
    "Epochs actually trained:",
    len(dropout_history.history["loss"])
)

Epochs actually trained: 90


In [17]:
dropout_train_loss, dropout_train_acc = dropout_model.evaluate(
    X_train_processed,
    y_train,
    verbose=0
)

dropout_val_loss, dropout_val_acc = dropout_model.evaluate(
    X_val_processed,
    y_val,
    verbose=0
)

print(f"Training Accuracy: {dropout_train_acc * 100:.2f}%")
print(f"Validation Accuracy: {dropout_val_acc * 100:.2f}%")
print(f"Training Loss: {dropout_train_loss:.4f}")
print(f"Validation Loss: {dropout_val_loss:.4f}")

Training Accuracy: 75.71%
Validation Accuracy: 60.00%
Training Loss: 0.4833
Validation Loss: 0.5023


In [18]:
l2_model = Sequential([
    tf.keras.Input(shape=(input_size,)),

    Dense(
        32,
        activation="relu",
        kernel_regularizer=l2(0.001)
    ),
    Dropout(0.30),

    Dense(
        16,
        activation="relu",
        kernel_regularizer=l2(0.001)
    ),
    Dropout(0.20),

    Dense(3, activation="softmax")
])

In [19]:
l2_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [20]:
l2_early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [21]:
l2_history = l2_model.fit(
    X_train_processed,
    y_train,
    validation_data=(X_val_processed, y_val),
    epochs=200,
    batch_size=16,
    callbacks=[l2_early_stopping],
    verbose=0
)

In [22]:
l2_train_loss, l2_train_acc = l2_model.evaluate(
    X_train_processed,
    y_train,
    verbose=0
)

l2_val_loss, l2_val_acc = l2_model.evaluate(
    X_val_processed,
    y_val,
    verbose=0
)

print(f"Training Accuracy: {l2_train_acc * 100:.2f}%")
print(f"Validation Accuracy: {l2_val_acc * 100:.2f}%")
print(f"Training Loss: {l2_train_loss:.4f}")
print(f"Validation Loss: {l2_val_loss:.4f}")

Training Accuracy: 72.86%
Validation Accuracy: 53.33%
Training Loss: 0.5378
Validation Loss: 0.5738


In [23]:
regularization_results = pd.DataFrame({
    "Model": [
        "Medium Baseline",
        "Early Stopping",
        "Dropout + Early Stopping",
        "L2 + Dropout + Early Stopping"
    ],

    "Training Accuracy": [
        90.71,
        early_train_acc * 100,
        dropout_train_acc * 100,
        l2_train_acc * 100
    ],

    "Validation Accuracy": [
        70.00,
        early_val_acc * 100,
        dropout_val_acc * 100,
        l2_val_acc * 100
    ],

    "Training Loss": [
        0.23,
        early_train_loss,
        dropout_train_loss,
        l2_train_loss
    ],

    "Validation Loss": [
        0.77,
        early_val_loss,
        dropout_val_loss,
        l2_val_loss
    ]
})

regularization_results.round(2)

,Model,Training Accuracy,Validation Accuracy,Training Loss,Validation Loss
0,Medium Baseline,90.71,70.00,0.23,0.77
1,Early Stopping,78.57,60.00,0.44,0.53
2,Dropout + Early Stopping,75.71,60.00,0.48,0.50
3,L2 + Dropout + Early Stopping,72.86,53.33,0.54,0.57


In [24]:
print(
    regularization_results
    .round(2)
    .to_string(index=False)
)

                        Model  Training Accuracy  Validation Accuracy  Training Loss  Validation Loss
              Medium Baseline              90.71                70.00           0.23             0.77
               Early Stopping              78.57                60.00           0.44             0.53
     Dropout + Early Stopping              75.71                60.00           0.48             0.50
L2 + Dropout + Early Stopping              72.86                53.33           0.54             0.57
